# 🍷 Wine Quality Prediction - Version 3 (Advanced)

## 🎯 Goals
1. **Advanced Feature Engineering**: Log-transformations, correlation filtering, and feature selection.
2. **Multi-Model Training**: RandomForest, LightGBM, XGBoost, CatBoost, SVR, KNN Regressor.
3. **Robust Evaluation**: 5-Fold Cross-Validation for stable RMSE and R² estimation.
4. **Advanced Ensemble**: Weighted average & stacking of top-performing regressors.
5. **Target Metric**: Achieve lower RMSE / higher R² on validation and test sets.


In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
import json
# Add src to path
sys.path.append(os.path.abspath("src"))


from src.preprocess.feature.feature import Feature
from src.pipeline.BasePipeline import BasePipelineReg


from model.Random_Forest import ModelRandomForest
from model.LightGBM import ModelLightGBM
from model.XGBoost import ModelXGBoost
from model.Catboost import ModelCatboost


from model.SVM import ModelSVM        
from model.KNN import ModelKNN       


from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

## 1. Load Data

In [2]:

DATA_DIR = "data"
file_path = os.path.join(DATA_DIR, "winequality-red.csv")

columns = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
    'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide',
    'density', 'pH', 'sulphates', 'alcohol', 'quality'
]

df = pd.read_csv(
    file_path,
    sep=',',          
    header=None,
    names=columns
)

print(f"Data shape: {df.shape}")
display(df.head())


X = df.drop('quality', axis=1)
y = df['quality']

print("X shape:", X.shape)
print("y shape:", y.shape)

Data shape: (1599, 12)


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


X shape: (1599, 11)
y shape: (1599,)


## 2. Feature Engineering & Preprocessing

In [3]:
numeric_cols = X.select_dtypes(include=np.number).columns

X_processed = X.copy()

for col in numeric_cols:
    median_val = X_processed[col].median()
    X_processed[col] = X_processed[col].fillna(median_val)


skewed_features = ['residual_sugar', 'chlorides', 'total_sulfur_dioxide']

for col in skewed_features:
    if col in X_processed.columns:
        min_val = X_processed[col].min()
        shift = abs(min_val) + 1 if min_val <= 0 else 0
        X_processed[f'{col}_log'] = np.log1p(X_processed[col] + shift)


corr_matrix = X_processed.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
print(f"Columns to drop: {to_drop}")

X_processed = X_processed.drop(columns=to_drop)

print(f"Processed shape: {X_processed.shape}")

Columns to drop: ['residual_sugar_log', 'chlorides_log']
Processed shape: (1599, 12)


### Feature Selection

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n===== Fold {fold+1} =====")

    # Split
    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # =========================
    # Feature Engineering
    # =========================
    ft = Feature()

    X_train = ft.transform(X_train)
    X_val = ft.transform(X_val)

    # Align columns giữa train và val (QUAN TRỌNG)
    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

    print("Train shape:", X_train.shape)
    print("Val shape:", X_val.shape)

    # (model sẽ thêm ở bước sau)


===== Fold 1 =====
Train shape: (1279, 11)
Val shape: (320, 11)

===== Fold 2 =====
Train shape: (1279, 11)
Val shape: (320, 11)

===== Fold 3 =====
Train shape: (1279, 11)
Val shape: (320, 11)

===== Fold 4 =====
Train shape: (1279, 11)
Val shape: (320, 11)

===== Fold 5 =====
Train shape: (1280, 11)
Val shape: (319, 11)


## 3. Model Training (5-Fold CV)
Train multiple regressors (RF, LightGBM, XGBoost, CatBoost, SVR, KNN) using 5-Fold CV for robust evaluation.

In [10]:

models = {
    "RandomForest": ModelRandomForest,  # class, không có ()
    "LightGBM": ModelLightGBM,
    "XGBoost": ModelXGBoost,
    "CatBoost": ModelCatboost
}
oof_preds = {}
model_metrics = {}


for name, model in models.items():
    print(f"\n{'='*20} Training {name} {'='*20}")
    pipeline = BasePipelineReg(model)
    df_train = X.copy()
    df_train['quality'] = y 
    oof, test_preds, metrics = pipeline.run_cv(df, target_col='quality')
    
    oof_preds[name] = oof
    model_metrics[name] = metrics
    
    print(f"{name} RMSE: {metrics['RMSE']:.4f} | MAE: {metrics['MAE']:.4f} | R2: {metrics['R2']:.4f}")


==================== Training RandomForest ====================

🔥 Fold 1
   Fold RMSE: 0.5722

🔥 Fold 2
   Fold RMSE: 0.6039

🔥 Fold 3
   Fold RMSE: 0.6081

🔥 Fold 4
   Fold RMSE: 0.5997

🔥 Fold 5
   Fold RMSE: 0.5508

📊 FINAL CV RESULT - RMSE: 0.5874 | R2: 0.4706
RandomForest RMSE: 0.5874 | MAE: 0.4388 | R2: 0.4706

==================== Training LightGBM ====================

🔥 Fold 1
   Fold RMSE: 0.5720

🔥 Fold 2
   Fold RMSE: 0.6168

🔥 Fold 3
   Fold RMSE: 0.6014

🔥 Fold 4
   Fold RMSE: 0.6299

🔥 Fold 5
   Fold RMSE: 0.5737

📊 FINAL CV RESULT - RMSE: 0.5992 | R2: 0.4491
LightGBM RMSE: 0.5992 | MAE: 0.4342 | R2: 0.4491

==================== Training XGBoost ====================

🔥 Fold 1
   Fold RMSE: 0.5815

🔥 Fold 2
   Fold RMSE: 0.6151

🔥 Fold 3
   Fold RMSE: 0.5848

🔥 Fold 4
   Fold RMSE: 0.6030

🔥 Fold 5
   Fold RMSE: 0.5582

📊 FINAL CV RESULT - RMSE: 0.5889 | R2: 0.4679
XGBoost RMSE: 0.5889 | MAE: 0.4033 | R2: 0.4679

==================== Training CatBoost ==================

## 4. Model Comparison

In [11]:
metrics_df = pd.DataFrame(model_metrics).T

# Sort theo RMSE (càng nhỏ càng tốt)
metrics_df = metrics_df.sort_values("RMSE", ascending=True)

display(metrics_df[['RMSE', 'MAE', 'R2']])

,RMSE,MAE,R2
RandomForest,0.587390,0.438835,0.470623
XGBoost,0.588880,0.403255,0.467933
CatBoost,0.590922,0.441865,0.464238
LightGBM,0.599199,0.434184,0.449124


## 5. Ensemble Strategy

### A. Weighted Average
Find optimal weights for the top-performing regressors to minimize RMSE on validation set.

In [12]:

top_models = metrics_df.head(5).index.tolist()
print(f"Top models for ensemble: {top_models}")


def get_ensemble_score(weights):
    weights = np.array(weights)
    weights /= weights.sum()

    # Weighted average predictions
    final_pred = np.zeros_like(oof_preds[top_models[0]])

    for i, model_name in enumerate(top_models):
        final_pred += weights[i] * oof_preds[model_name]

    rmse = np.sqrt(mean_squared_error(y, final_pred))
    return rmse  # minimize RMSE


init_weights = [1 / len(top_models)] * len(top_models)
bounds = [(0, 1)] * len(top_models)

res = minimize(get_ensemble_score, init_weights, bounds=bounds, method='SLSQP')

best_weights = res.x / res.x.sum()

print("\n🔥 Best Weights:")
for m, w in zip(top_models, best_weights):
    print(f"{m}: {w:.4f}")

print(f"\n📊 Optimized Ensemble RMSE: {res.fun:.4f}")

Top models for ensemble: ['RandomForest', 'XGBoost', 'CatBoost', 'LightGBM']

🔥 Best Weights:
RandomForest: 0.3497
XGBoost: 0.4444
CatBoost: 0.2059
LightGBM: 0.0000

📊 Optimized Ensemble RMSE: 0.5763


### B. Stacking (Logistic Regression Meta-Model)

In [13]:
X_meta_train = []

for name in top_models:
    X_meta_train.append(oof_preds[name].reshape(-1, 1))

X_meta_train = np.hstack(X_meta_train)

print(f"Meta Train Shape: {X_meta_train.shape}")


meta_model = LinearRegression()
meta_model.fit(X_meta_train, y)


meta_train_pred = meta_model.predict(X_meta_train)

rmse = np.sqrt(mean_squared_error(y, meta_train_pred))
r2 = r2_score(y, meta_train_pred)

print(f"🔥 Stacking RMSE: {rmse:.4f}")
print(f"📊 Stacking R2: {r2:.4f}")

Meta Train Shape: (1599, 4)
🔥 Stacking RMSE: 0.5761
📊 Stacking R2: 0.4908


## 6. Final Prediction & Submission

In [14]:

final_pred = np.zeros(len(X_processed))

for i, name in enumerate(top_models):
  
    final_pred += best_weights[i] * oof_preds[name]


submission = pd.DataFrame({
    'Id': range(len(final_pred)),
    'quality': final_pred
})



os.makedirs("experiments/Wine_V3", exist_ok=True)
sub_path = "experiments/Wine_V3/submission_wine_v3_demo.csv"
submission.to_csv(sub_path, index=False)

print(f"✅ Submission demo saved to {sub_path}")
submission.head()

✅ Submission demo saved to experiments/Wine_V3/submission_wine_v3_demo.csv


,Id,quality
0,0,5.091039
1,1,4.861836
2,2,5.288093
3,3,5.474582
4,4,5.073141


In [15]:

os.makedirs("experiments/Wine_V3", exist_ok=True)


artifacts = {
    "best_weights": {m: w for m, w in zip(top_models, best_weights)},
    "metrics": model_metrics,                # dùng model_metrics đã lưu trong training loop
    "selected_features": X_processed.columns.tolist()  # dùng X_processed sau feature engineering
}


with open("experiments/Wine_V3/artifacts_v3.json", "w") as f:
    json.dump(artifacts, f, indent=4)

print("✅ Artifacts saved to experiments/Wine_V3/artifacts_v3.json")

✅ Artifacts saved to experiments/Wine_V3/artifacts_v3.json
